<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml(
        name='mnist_784',
        version=1,
        as_frame=False,
        return_X_y=True
    )
    
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y  
    )
    
    return X_train, X_test, y_train, y_test

# Teste
X_train, X_test, y_train, y_test = load_data(seed=42)

print("Formato X_train :", X_train.shape)
print("Formato X_test  :", X_test.shape)
print("Formato y_train :", y_train.shape)
print("Formato y_test  :", y_test.shape)

Formato X_train : (56000, 784)
Formato X_test  : (14000, 784)
Formato y_train : (56000,)
Formato y_test  : (14000,)


1. Não, pois modelos de árvore de decisão são invariáveis a escala

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(
        n_estimators=100,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [30]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    return acc

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [42]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
 
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    
    return acc

run_pipeline()

0.9672142857142857

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [32]:
from sklearn.tree import DecisionTreeClassifier

profundidades = [1, 3, 5, 10, 15, 20, None]

print(f"{'max_depth':<12} {'Treino':>8} {'Teste':>8}")
print("-" * 30)
for depth in profundidades:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    acc_treino = accuracy_score(y_train, tree.predict(X_train))
    acc_teste  = accuracy_score(y_test,  tree.predict(X_test))
    print(f"{str(depth):<12} {acc_treino:>8.4f} {acc_teste:>8.4f}")

max_depth      Treino    Teste
------------------------------
1              0.1989   0.1982
3              0.4422   0.4444
5              0.6591   0.6554
10             0.9057   0.8613
15             0.9868   0.8812
20             0.9951   0.8768
None           1.0000   0.8742


Após passar de 15 é possível notar uma queda na acurácia de teste, enquanto a acurácia de treino cresce.
Quando é None a árvore está completamente moldada para o grupo de teste, criando folhas muito específicas para cada entrada de dado.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_ada = run_pipeline("ab")
accuracy_rand = run_pipeline("rf")

print("Acurácia AdaBoost:", accuracy_ada)
print("Acurácia Random Forest:", accuracy_rand)

X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model  = train_random_forest(X_train, y_train, seed=42)
ada_model = train_adaboost(X_train, y_train, seed=42)

print(f"\n{'Métrica':<12} {'Random Forest':>15} {'AdaBoost':>12}")
print("-" * 42)

for nome, model in [("Random Forest", rf_model), ("AdaBoost", ada_model)]:
    y_pred = model.predict(X_test)
    print(f"\n{nome}")
    print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  Recall    : {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  F1-Score  : {f1_score(y_test, y_pred, average='weighted'):.4f}")

Acurácia AdaBoost: 0.7387142857142858
Acurácia Random Forest: 0.9672142857142857

Métrica        Random Forest     AdaBoost
------------------------------------------

Random Forest
  Accuracy  : 0.9672
  Precision : 0.9672
  Recall    : 0.9672
  F1-Score  : 0.9672

AdaBoost
  Accuracy  : 0.7387
  Precision : 0.7479
  Recall    : 0.7387
  F1-Score  : 0.7412


O melhor modelo apresentado foi o Random Forest.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [38]:
accuracy_ada = run_pipeline("ab")
accuracy_rand = run_pipeline("rf")

print("Acurácia AdaBoost com Seed=42:", accuracy_ada)
print("Acurácia Random Forest com Seed=42:", accuracy_rand)

accuracy_ada = run_pipeline("ab", 7)
accuracy_rand = run_pipeline("rf", 7)

print("Acurácia AdaBoost com Seed=7:", accuracy_ada)
print("Acurácia Random Forest com Seed=7:", accuracy_rand)

Acurácia AdaBoost com Seed=42: 0.7387142857142858
Acurácia Random Forest com Seed=42: 0.9672142857142857
Acurácia AdaBoost com Seed=7: 0.7177142857142857
Acurácia Random Forest com Seed=7: 0.9681428571428572


Com a Seed=7 a acurácia do Random Forest melhorou muito pouco (0,001) enquanto no AdaBoost deu ume leve piorada (0,02).

Sim. Ao fixar uma mesma seed, todos os elementos aleatórios do pipeline são controlados

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [39]:
acc_treino = accuracy_score(y_train, rf_model.predict(X_train))
acc_teste  = accuracy_score(y_test,  rf_model.predict(X_test))

print("Acurácia no Treino:", acc_treino)
print("Acurácia no Teste :", acc_teste)
print("Diferença :", round(acc_treino - acc_teste, 4))

Acurácia no Treino: 1.0
Acurácia no Teste : 0.9672142857142857
Diferença : 0.0328


Sim, aacurácia 100% no treino contra 96.72% no teste indica que o modelo memorizou os dados de treino

O Random Forest mitiga o problema via bagging, mas o AdaBoost pode ser mais problemático em datasets com ruído, pois o boosting sequencial aumenta o peso de amostras difíceis, podendo memorizar outliers.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
estimators = [10, 50, 100, 200]

print(f"{'n_estimators':<15} {'Random Forest':>15} {'AdaBoost':>12}")
print("-" * 44)

for n in estimators:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    acc_rf = accuracy_score(y_test, rf.predict(X_test))

    ada = AdaBoostClassifier(n_estimators=n, random_state=42)
    ada.fit(X_train, y_train)
    acc_ada = accuracy_score(y_test, ada.predict(X_test))

    print(f"{n:<15} {acc_rf:>15.4f} {acc_ada:>12.4f}")

n_estimators      Random Forest     AdaBoost
--------------------------------------------
10                       0.9464       0.3337
50                       0.9639       0.6681
100                      0.9672       0.7387
200                      0.9682       0.7684


No Random Forest a mudança é muito pequena enquanto no AdaBoost há uma mudança significativa, evidenciado na mudança de 10 para 50 estimadores

O AdaBoost é o mais sensível

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
# TODO: implemente load_data

1. Não,a acurácia sozinha esconde onde o modelo erra. Métricas como Precision, Recall e F1-Score, além da Matriz de Confusão, são essenciais para expor falhas específicas em cada classe que a média global mascara.

2. O random_state fixo garante a reprodutibilidade (mesmo resultado ao rodar de novo)

3. 
- Divisão única (Train/Test Split): O resultado fica refém de uma única partição. Se a divisão der sorte e pegar exemplos "fáceis" no teste, o desempenho será superestimado.

- Vazamento no Ajuste (Data Leakage): Escolher o melhor n_estimators olhando direto para o resultado do teste "vicia" o modelo. O correto seria usar um conjunto de validação para o ajuste.

4. Parcialmente. Ele é frágil por não usar validação cruzada nem uma etapa de validação separada para os parâmetros. Para um experimento rigoroso, precisaria de uma avaliação mais robusta que uma única rodada de treino e teste.